# Small vs Big : Qwen/Kimi vs modèles frontier
Comparaison des modèles plus légers (Qwen, Kimi) et des gros modèles frontier : paramètres, compute, coût, énergie (quand dispo).


**Objectif** : Avoir des graphiques prêts pour les slides : volumes de paramètres, compute, coûts, énergie. Les métriques de performance ne sont pas présentes dans les CSV, donc non comparées ici.
Données : `frontier_ai_models.csv`, `all_ai_models.csv`, `notable_ai_models.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA_DIR = Path("..") / "data" / "ai_models"


In [ ]:
def parse_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    s = str(value).lower()
    s = s.replace(",", "").replace(" ", "").replace("~", "").replace("≈", "")
    s = s.replace("×10^", "e").replace("x10^", "e").replace("×10", "e").replace("x10", "e")
    s = s.replace("^", "e").replace(">", "").replace("<", "")
    m = re.match(r"([0-9.+\-e]+)([kmbt]?)", s)
    if not m:
        return np.nan
    num, suf = m.groups()
    try:
        base = float(num)
    except ValueError:
        return np.nan
    mult = {"k": 1e3, "m": 1e6, "b": 1e9, "t": 1e12}.get(suf, 1)
    return base * mult


In [ ]:
files = {
    "frontier": pd.read_csv(DATA_DIR / "frontier_ai_models.csv"),
    "all": pd.read_csv(DATA_DIR / "all_ai_models.csv"),
    "notable": pd.read_csv(DATA_DIR / "notable_ai_models.csv"),
}

def normalize(df, source_name):
    df = df.copy()
    df["publication_year"] = pd.to_datetime(df.get("Publication date"), errors="coerce").dt.year
    mapping = {
        "Training compute (FLOP)": "compute_flop",
        "Parameters": "parameters",
        "Training dataset size (gradients)": "train_tokens",
        "Training compute cost (2023 USD)": "train_cost_usd",
        "Training power draw (W)": "power_w",
        "Training time (hours)": "train_hours",
    }
    for src, tgt in mapping.items():
        if src in df.columns:
            df[tgt] = df[src].apply(parse_number)
        else:
            df[tgt] = np.nan
    df["source"] = source_name
    return df

norm = {name: normalize(d, name) for name, d in files.items()}

small_names = ["qwen", "kimi", "moonshot", "moonshotkimi", "qwen2", "qwen3", "qwen-2", "qwen-3"]
pattern = re.compile("|".join([re.escape(n) for n in small_names]), re.IGNORECASE)

small_df = pd.concat([
    n[n["Model"].fillna("").str.contains(pattern)]
    for n in norm.values()
], ignore_index=True)

# Drop duplicates on Model to avoid double counting the same entries across files
small_df = small_df.drop_duplicates(subset=["Model"], keep="first")
small_df["segment"] = "Small (Qwen/Kimi)"

big_df = norm["frontier"].copy()
big_df = big_df.dropna(subset=["compute_flop"])
# garder les modèles top compute
big_df = big_df.sort_values("compute_flop", ascending=False).head(30)
big_df["segment"] = "Frontier (Big)"

cols = [
    "Model", "Organization", "publication_year", "parameters", "compute_flop",
    "train_cost_usd", "power_w", "train_hours", "train_tokens", "segment"
]
small = small_df[cols]
big = big_df[cols]
combined = pd.concat([small, big], ignore_index=True)

coverage = combined.groupby("segment")[
    ["parameters", "compute_flop", "train_cost_usd", "power_w", "train_hours"]
].count()
coverage


Les petites entrées (Qwen/Kimi) ont souvent des valeurs manquantes pour le coût et l'énergie; les grosses disposent davantage de compute et parfois de coût/puissance.


In [ ]:
# Boxplot des paramètres
fig, ax = plt.subplots()
combined.boxplot(column="parameters", by="segment", ax=ax)
ax.set_yscale("log")
ax.set_ylabel("Paramètres")
ax.set_title("Taille modèle : Small vs Big")
plt.suptitle("")
plt.tight_layout()
plt.show()


Les modèles frontier ont plusieurs ordres de grandeur de plus en paramètres que les entrées Qwen/Kimi listées ici.


In [ ]:
# Scatter compute vs paramètres
fig, ax = plt.subplots()
for seg, df_seg in combined.dropna(subset=["parameters", "compute_flop"]).groupby("segment"):
    ax.scatter(df_seg["parameters"], df_seg["compute_flop"], label=seg, alpha=0.75)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Paramètres")
ax.set_ylabel("Compute (FLOP)")
ax.set_title("Compute vs paramètres : Small vs Big")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()


Les points Big se trouvent nettement au-dessus : ils mobilisent bien plus de compute pour des tailles supérieures. Les petits restent dans une zone basse en compute.


In [ ]:
# Coût vs compute (lignes avec coût)
cost_avail = combined.dropna(subset=["compute_flop", "train_cost_usd"])
fig, ax = plt.subplots()
for seg, df_seg in cost_avail.groupby("segment"):
    ax.scatter(df_seg["compute_flop"], df_seg["train_cost_usd"], label=seg, alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Compute (FLOP)")
ax.set_ylabel("Coût (USD 2023)")
ax.set_title("Coût vs compute")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()


Les modèles Qwen/Kimi présents n'ont pas ou peu de coûts renseignés : la comparaison coût reste dominée par les modèles Big.


In [ ]:
# Energie estimée (power * heures)
energy_df = combined.dropna(subset=["power_w", "train_hours"])
energy_df["energy_mwh"] = energy_df["power_w"] * energy_df["train_hours"] / 1e6

fig, ax = plt.subplots()
for seg, df_seg in energy_df.groupby("segment"):
    ax.scatter(df_seg["publication_year"], df_seg["energy_mwh"], label=seg, alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel("Année")
ax.set_ylabel("Energie (MWh)")
ax.set_title("Energie d'entraînement estimée")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

energy_stats = energy_df.groupby("segment")["energy_mwh"].describe()
energy_stats


Les données énergie sont quasi absentes côté Qwen/Kimi dans ces CSV ; côté Big, on observe des MWh allant des centaines aux dizaines de milliers.


In [ ]:
# Barres : top 5 compute par segment
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, seg in zip(axes, ["Small (Qwen/Kimi)", "Frontier (Big)"]):
    subset = combined[combined["segment"] == seg].dropna(subset=["compute_flop"])
    top = subset.sort_values("compute_flop", ascending=False).head(5)
    ax.barh(top["Model"], top["compute_flop"], color="#4C72B0")
    ax.set_xscale("log")
    ax.set_title(f"Top compute - {seg}")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()


Les barres montrent l'écart massif : les top Small restent plusieurs ordres de grandeur en dessous des top Frontier.


## Synthèse
- **Paramètres** : les modèles frontier dominent largement en taille; Qwen/Kimi listés ici restent plus compacts.
- **Compute** : corrélé à la taille, les Small restent bas; les Big dépassent 1e24 FLOP.
- **Coûts** : peu de données côté Qwen/Kimi; côté Big, on est sur des centaines de M$ pour les plus récents.
- **Énergie** : champs quasi vides pour Qwen/Kimi dans ces CSV; les estimations disponibles côté Big montrent des entraînements à centaines/milliers de MWh.
- **Performance** : non fournie dans ces fichiers; pour juger l'efficience qualité/compute, il faut compléter avec des benchmarks publics.
- **À ajouter si disponible** : prix d'inférence, latence, consommation en production, contexte window (pour l'usage), et coûts cloud vs on-prem.
